# Basics of RSOME

RSOME is a Python package for formulating and solving optimization problems. It provides an array-oriented modeling interface that makes it convenient to express linear, mixed-integer, convex, and conic optimization models using familiar Python and NumPy operations.

This guide introduces model creation, decision variables, linear and convex constraints, matrix operations, standard forms, and solution methods. For installation instructions, see the [installation guide](rsome_install:pypi).

(rsome_basics:modeling_env)=
## Modeling Environment

The `rsome.demod` module provides the `Model` class for defining optimization problems. A model contains decision variables, an objective function, and constraints. The `rsome.math` module provides mathematical functions for constructing expressions and constraints.

(rsome_basics:intro_ro_env)=
## Introduction to the `rsome.demod` Environment

### Models

Create a model by calling `demod.Model()`:


In [1]:
from rsome import demod         # import the deterministic modeling tool

model = demod.Model()           # create a Model object


The code above creates a `Model` object, used to define decision variables, objectives, and constraints.

### Decision Variables and Linear Constraints

Use `model.var(shape, vtype)` to create decision-variable arrays. `shape` specifies the array dimensions; `vtype` is `'C'` for continuous variables (the default), `'B'` for binary variables, and `'I'` for general integers. For instance, the following statements create:

- `x`: a one-dimensional array of three integer variables;
- `y`: a $3\times5$ array of 15 binary variables;
- `z`: a $2\times3\times4\times5$ array of 120 continuous variables.


In [2]:
x = model.var(3, vtype='I')     # 3 integer variables as a 1D array
y = model.var((3, 5), 'B')      # 3x5 binary variables as a 2D array
z = model.var((2, 3, 4, 5))     # 2x3x4x5 continuous variables as a 4D array


In [3]:
import numpy as np

I, J = 7, 13

A = np.random.rand(J, I)
b = np.random.rand(I)
c = np.random.rand(J)

RSOME variable arrays support NumPy-like elementwise arithmetic, matrix multiplication, broadcasting, indexing, slicing, and reductions. The following constraints use $x\in\mathbb{R}^I$, $y\in\mathbb{R}^{J\times I}$, $A\in\mathbb{R}^{J\times I}$, $b\in\mathbb{R}^I$, and $c\in\mathbb{R}^J$:

$$
\begin{aligned}
 b^\top x &= 1, & Ax &\le c,\\
 \sum_{j,i}y_{ji} &\ge 1, & \sum_i y_{ji} &\ge 0\quad(j=1,\ldots,J),\\
 A_{ji}x_i &\ge 1, & A_{ji}y_{ji}+x_i &\ge 0.
\end{aligned}
$$

The next cell constructs these expressions but does not add them to the model until `model.st()` is called.


In [4]:
x = model.var(I)                # define x as a 1D array of I variables
y = model.var((J, I))           # define y as a 2D array of JxI variables

b @ x == 1
A @ x <= c
y.sum() >= 1
y.sum(axis=1) >= 0
A * x >= 1
A*y + x >= 0


91 linear inequality constraints

RSOME arrays can also be used to specify an objective function. An objective must be a scalar expression (its `size` must be one). A minimization objective may be convex or affine; a maximization objective may be concave or affine.

```python
model.min(b @ x)        # minimize b @ x
model.max(b @ x)        # maximize b @ x
```

Use one of these statements to set the objective of a model. Calling the other method sets a different objective.

Constraints are added with `model.st()`, short for *subject to*. The method accepts one or more constraint objects. A sequence produced by a generator can be unpacked into separate arguments, as illustrated below.


In [5]:
model.st(A @ x <= c)                    # define one constraint

model.st(y.sum() >= 1,
         y.sum(axis=1) >= 0,
         A*y + x >= 0)                  # define multiple constraints

model.st(*(x[i] <= i for i in range(3)))  # unpack constraints from a loop


### Convex Expressions and Convex Constraints

The `rsome.math` module supplies functions for constructing convex and concave expressions and conic constraints. In the following table, `x`, `y`, and `z` denote compatible RSOME expressions or numeric arguments.

| Function | Description |
|:--|:--|
| `abs(x)` | Elementwise absolute values. |
| `entropy(x)` | Entropy expression $-\sum_i x_i\log(x_i)$. |
| `exp(x)` | Elementwise natural exponential. |
| `expcone(x, y, z)` | Exponential-cone constraint $z\exp(x/z)\leq y$. |
| `kldiv(x, p)` | Kullback–Leibler divergence expression with reference values `p`. |
| `log(x)` | Elementwise natural logarithm. |
| `norm(x, degree=2)` | Vector norm, with the norm degree specified by `degree`. |
| `pexp(x, scale)` | Perspective exponential $\text{scale}\exp(x/\text{scale})$. |
| `plog(x, scale)` | Perspective logarithm $\text{scale}\log(x/\text{scale})$. |
| `quad(x, Q)` | Quadratic form involving the matrix `Q`. |
| `rsocone(x, y, z)` | Rotated second-order cone constraint. |
| `square(x)` | Elementwise squares. |
| `sumsqr(x)` | Sum of squares. |

### Matrix Operations

RSOME supports operations on matrix-valued expressions, including transposition, diagonals, triangular parts, and traces.

| Function | Description |
|:--|:--|
| `diag(x, fill=False)` | Extract diagonal elements; with `fill=True`, construct a diagonal matrix. |
| `trace(x)` | Trace of a square matrix expression. |
| `tril(x, k=0)` | Lower triangular part, with diagonal offset `k`. |
| `triu(x, k=0)` | Upper triangular part, with diagonal offset `k`. |
| `schur(A, B, C)` | Positive-semidefinite constraint on the symmetric block matrix formed by `A`, `B`, and `C`. |

The `.T` property transposes a matrix expression. For example, the following block matrix can be constrained to be positive semidefinite:

$$
\begin{pmatrix} A & Z \\ Z^\top & \operatorname{diag}(Z) \end{pmatrix} \succeq 0,
$$

where $A$ is an $n\times n$ matrix and $Z$ is a lower triangular matrix. The next cells express this constraint using `tril()`, `diag()`, and `schur()`.


In [6]:
n = 5

In [7]:
from rsome import math as rso
from rsome import demod

model = demod.Model()

A = model.var((n, n))
Z = rso.tril(model.var((n, n)))

model.st(rso.schur(A, Z, rso.diag(Z, fill=True)))


The `T` attribute of a two-dimensional RSOME expression follows NumPy's transpose convention. The `schur` function constructs a block positive-semidefinite constraint; it does not numerically evaluate the matrix.

(rsome_basics:sform_sol)=
## Standard Forms and Solutions

RSOME converts the model into a conic optimization program for a solver. Use `model.do_math()` for the primal standard form or `model.do_math(primal=False)` for the dual standard form.


In [8]:
from rsome import demod
from rsome import math as rso
import numpy as np

n = 3
np.random.seed(1)
c = np.random.normal(size=n)

model = demod.Model()
x = model.var(n)

model.max(c @ x)
model.st(rso.norm(x) <= 1)

primal = model.do_math()            # primal conic standard form
dual = model.do_math(primal=False)  # dual conic standard form


In [9]:
primal

Number of variables:           7
Continuous/binaries/integers:  7/0/0
---------------------------------------------
Number of linear constraints:  4
Inequalities/equalities:       1/3
Number of coefficients:        7
---------------------------------------------
Number of SOC constraints:     1
---------------------------------------------
Number of ExpCone constraints: 0
---------------------------------------------
Number of PSCone constraints:  0

In [10]:
dual

Number of variables:           4
Continuous/binaries/integers:  4/0/0
---------------------------------------------
Number of linear constraints:  3
Inequalities/equalities:       0/3
Number of coefficients:        3
---------------------------------------------
Number of SOC constraints:     1
---------------------------------------------
Number of ExpCone constraints: 0
---------------------------------------------
Number of PSCone constraints:  0

The `show()` method of a standard-form program returns a Pandas DataFrame describing the variables, objective coefficients, and constraints.


In [11]:
primal.show()

,x1,x2,x3,x4,x5,x6,x7,Sense,Constant
Obj,-1.624345,0.611756,0.528172,-0.0,-0.0,-0.0,-0.0,-,-
LC1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,<=,1.0
LC2,-1.0,0.0,0.0,1.0,0.0,0.0,0.0,==,-0.0
LC3,0.0,-1.0,0.0,0.0,1.0,0.0,0.0,==,-0.0
LC4,0.0,0.0,-1.0,0.0,0.0,1.0,0.0,==,-0.0
QC1,0.0,0.0,0.0,1.0,1.0,1.0,-1.0,-,-
UB,inf,inf,inf,inf,inf,inf,inf,-,-
LB,-inf,-inf,-inf,-inf,-inf,-inf,0.0,-,-
Type,C,C,C,C,C,C,C,-,-


In [12]:
dual.show()

,x1,x2,x3,x4,Sense,Constant
Obj,1.0,0.0,0.0,0.0,-,-
LC1,0.0,-1.0,0.0,0.0,==,-1.624345
LC2,0.0,0.0,-1.0,0.0,==,0.611756
LC3,0.0,0.0,0.0,-1.0,==,0.528172
QC1,-1.0,1.0,1.0,1.0,-,-
UB,inf,inf,inf,inf,-,-
LB,0.0,-inf,-inf,-inf,-,-
Type,C,C,C,C,-,-


A model can be solved with `model.solve(solver=None, display=True, log=False, params={})`. The `solver` argument identifies the optimization solver. When it is omitted, RSOME selects a default solver based on the model's properties.

| Solver | Solver name | SOCP | ECP | SDP |
|:--|:--|:--|:--|:--|
| SciPy | `'scipy'` | No | No | No |
| ECOS | `'ecos'` | Yes | Yes | No |
| Clarabel | `'clarabel'` | Yes | Yes | Yes |
| Gurobi | `'gurobi'` | Yes | No | No |
| MOSEK | `'mosek'` | Yes | Yes | Yes |
| COPT | `'copt'` | Yes | No | No |

The example above contains a second-order cone constraint, so ECOS and Clarabel are suitable choices. Install the corresponding solver package before running its example. A solver can be selected by name or by passing its imported interface module.


In [13]:
from rsome.solvers import eco_solver as eco
from rsome.solvers import clb_solver as clb
from rsome.solvers import grb_solver as grb
from rsome.solvers import msk_solver as msk
from rsome.solvers import cpt_solver as cpt

The following calls solve the model using ECOS and Clarabel, respectively. Run the cell for a solver that is installed in your Python environment.


In [14]:
model.solve('ecos')


Being solved by ECOS 2.0.14...
Solution status: Optimal solution found
Running time: 0.0001s


In [15]:
model.solve('clarabel')


Being solved by Clarabel 0.11.1...
Solution status: Solved
Running time: 0.0001s


By default, RSOME displays the solver status and solution time. Set `display=False` to suppress this summary, or `log=True` to display solver logs.

Solver-specific settings can be supplied through the `params` dictionary. The dictionary keys are parameter names recognized by the selected solver, and the values specify their settings. The following examples show how to set a relative optimality-gap tolerance for Gurobi, MOSEK, and COPT. These examples require the corresponding solver installations and, where applicable, licenses.

For additional parameter settings, consult the [Gurobi parameter reference](https://docs.gurobi.com/projects/optimizer/en/current/reference/parameters.html), [MOSEK parameter reference](https://docs.mosek.com/latest/pythonapi/parameters.html), and [COPT documentation](https://www.shanshu.ai/copt).


In [16]:
# Run the call corresponding to an installed solver:
# model.solve('gurobi', params={'MIPGap': 1e-3})
# model.solve('mosek', params={'mioTolRelGap': 1e-3})
# model.solve('copt', params={'RelGap': 1e-3})


After a model has been solved successfully, `model.get()` returns its optimal objective value and `x.get()` returns the optimal value of the decision-variable array `x`. If an optimal solution is not available, these methods may raise an error.

```python
model.get()   # optimal objective value
x.get()       # optimal decision-variable values
```

(rsome_basics:examples)=
## Application Examples

The following examples illustrate optimization models that can be formulated with RSOME:

- [Mean-Variance Portfolio](examples/example_mean_variance_portfolio)
- [Integer Programming for Sudoku](examples/example_sudoku.ipynb)
- [Optimal DC Power Flow](examples/example_dc_opf.ipynb)
- [Conic Relaxation of Optimal AC Power Flows](examples/example_ac_opf.ipynb)
- [The Unit Commitment Problem](examples/example_uc.ipynb)
- [Box with the Maximum Volume](examples/example_max_box.ipynb)
- [Minimal Enclosing Ellipsoid](examples/example_min_ellipsoid.ipynb)
- [Logistic Regression](examples/example_logit.ipynb)


## References

For more information on RSOME's modeling functions and solver interfaces, consult the package documentation and the docstrings of the corresponding modules.


```{toctree}
:hidden:
:maxdepth: 1

examples/example_mean_variance_portfolio.ipynb
examples/example_sudoku.ipynb
examples/example_dc_opf.ipynb
examples/example_ac_opf.ipynb
examples/example_uc.ipynb
examples/example_max_box.ipynb
examples/example_min_ellipsoid.ipynb
examples/example_logit.ipynb
```